# AST Extraction

## 1. Imports

In [1]:
import pandas as pd
import re

## 2. Data processing

### 2.1 Load dataframes

In [2]:
df_plag_train = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_train.csv")
df_plag_test = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_test.csv")
df_plag_val = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_val.csv")

df_ai_train = pd.read_csv("../Dataframes/df_ai/df_ai_train.csv")
df_ai_test = pd.read_csv("../Dataframes/df_ai/df_ai_test.csv")
df_ai_val = pd.read_csv("../Dataframes/df_ai/df_ai_val.csv")

### 2.2 Estimate token lengths

In [3]:
for df in [df_plag_train, df_plag_test, df_plag_val]:
    df["approx_tokens_code1"] = df["code1"].apply(lambda x: len(str(x).split()))
    df["approx_tokens_code2"] = df["code2"].apply(lambda x: len(str(x).split()))

In [4]:
df_plag_train["approx_tokens_code1"].describe().round(2)

count    24000.00
mean       111.00
std        189.32
min          4.00
25%         37.00
50%         64.00
75%        116.00
max       4285.00
Name: approx_tokens_code1, dtype: float64

In [5]:
print(f"There are {(df_plag_train['approx_tokens_code1'] > 512).sum()} codes > 512 tokens in code 1 train set"
        f" ({(df_plag_train['approx_tokens_code1'] > 512).mean():.1%})")

There are 553 codes > 512 tokens in code 1 train set (2.3%)


In [6]:
df_plag_train["approx_tokens_code2"].describe().round(2)

count    24000.00
mean        72.53
std        117.60
min          7.00
25%         27.00
50%         42.00
75%         78.00
max       3494.00
Name: approx_tokens_code2, dtype: float64

In [7]:
print(f"There are {(df_plag_train['approx_tokens_code2'] > 512).sum()} codes > 512 tokens in code 2 train set"
        f" ({(df_plag_train['approx_tokens_code2'] > 512).mean():.1%})")

There are 222 codes > 512 tokens in code 2 train set (0.9%)


In [8]:
for df in [df_ai_train, df_ai_test, df_ai_val]:
    df["approx_tokens"] = df["code"].apply(lambda x: len(str(x).split()))

df_ai_train["approx_tokens"].describe().round(2)

count    24000.00
mean        46.55
std         56.48
min          6.00
25%         21.00
50%         33.00
75%         55.00
max       3284.00
Name: approx_tokens, dtype: float64

In [9]:
print(f"There are {(df_ai_train['approx_tokens'] > 512).sum()} codes > 512 tokens in ai train set"
        f" ({(df_ai_train['approx_tokens'] > 512).mean():.1%})")

There are 27 codes > 512 tokens in ai train set (0.1%)


## 3. Lexical analysis

### 3.1 Set up lexer

In [10]:
from enum import Enum, auto

class TokenType(Enum):
    KEYWORD = auto()
    ID = auto()
    STRING = auto()
    NUMBER = auto()
    OPEN_BRACE = auto()
    CLOSE_BRACE = auto()
    OPEN_PAREN = auto()
    CLOSE_PAREN = auto()
    COMMENT = auto()
    OTHER = auto()

class Token:
    def __init__(self, type: TokenType, value: str):
        self.type  = type
        self.value = value

    def __repr__(self):
        return f"Token({self.type.name}, {self.value!r})"

RESERVED = {
    "if", "else", "for", "while", "switch", "case",
    "class", "interface", "enum",
    "return", "new", "void", "static", "public",
    "private", "protected", "extends", "implements"
}

In [11]:
def lex(code: str) -> list:
    tokens = []
    i = 0
    n = len(code)

    while i < n:
        if code[i].isspace():
            i += 1
            continue

        if code[i:i+2] == "//":
            end = code.find("\n", i)
            end = end if end != -1 else n
            tokens.append(Token(TokenType.COMMENT, code[i:end]))
            i = end
            continue

        if code[i:i+2] == "/*":
            end = code.find("*/", i+2)
            end = end + 2 if end != -1 else n
            tokens.append(Token(TokenType.COMMENT, code[i:end]))
            i = end
            continue

        if code[i] == '"':
            j = i + 1
            while j < n and code[j] != '"':
                if code[j] == '\\':
                    j += 1
                j += 1
            tokens.append(Token(TokenType.STRING, code[i:j+1]))
            i = j + 1
            continue

        if code[i].isdigit():
            j = i
            while j < n and (code[j].isdigit() or code[j] == '.'):
                j += 1
            tokens.append(Token(TokenType.NUMBER, code[i:j]))
            i = j
            continue

        if code[i].isalpha() or code[i] == '_':
            j = i
            while j < n and (code[j].isalnum() or code[j] == '_'):
                j += 1
            word = code[i:j]
            ttype = TokenType.KEYWORD if word in RESERVED else TokenType.ID
            tokens.append(Token(ttype, word))
            i = j
            continue

        if code[i] == '{':
            tokens.append(Token(TokenType.OPEN_BRACE,  '{'))
            i += 1
            continue
        if code[i] == '}':
            tokens.append(Token(TokenType.CLOSE_BRACE, '}'))
            i += 1
            continue
        if code[i] == '(':
            tokens.append(Token(TokenType.OPEN_PAREN,  '('))
            i += 1
            continue
        if code[i] == ')':
            tokens.append(Token(TokenType.CLOSE_PAREN, ')'))
            i += 1
            continue

        tokens.append(Token(TokenType.OTHER, code[i]))
        i += 1

    return tokens

### 3.2 Extract stylometric features

In [12]:
def stylometric_analysis(code):
    lines = code.splitlines()
    num_lines = len(lines)

    num_comments = 0
    for line in lines:
        if line.strip().startswith("//") or line.strip().startswith("*") or line.strip().startswith("/*"):
            num_comments += 1
    comment_density = num_comments / num_lines

    non_empty_lines = []
    for line in lines:
        if line.strip():
            non_empty_lines.append(line)

    line_lengths = []
    for line in non_empty_lines:
        line_lengths.append(len(line))

    if line_lengths:
        avg_line_length = sum(line_lengths) / len(line_lengths)
    else:
        avg_line_length = 0.0

    if len(line_lengths) > 1:
        mean = avg_line_length
        line_length_variance = sum((l - mean) ** 2 for l in line_lengths) / len(line_lengths)
    else:
        line_length_variance = 0.0

    blank_lines = 0
    for line in lines:
        if not line.strip():
            blank_lines += 1
    blank_line_ratio = blank_lines / num_lines

    return {
        "comment_density": comment_density,
        "avg_line_length": avg_line_length,
        "line_length_variance": line_length_variance,
        "blank_line_ratio": blank_line_ratio,
    }

### 3.3 Store features in dataframes

In [13]:
for c in ["code1", "code2"]:
    df_plag_train[[f"comment_density_{c}", f"avg_line_length_{c}", f"line_length_variance_{c}", f"blank_line_ratio_{c}"]] = df_plag_train[c].apply(stylometric_analysis).apply(pd.Series)
    df_plag_val[[f"comment_density_{c}", f"avg_line_length_{c}", f"line_length_variance_{c}", f"blank_line_ratio_{c}"]] = df_plag_val[c].apply(stylometric_analysis).apply(pd.Series)
    df_plag_test[[f"comment_density_{c}", f"avg_line_length_{c}", f"line_length_variance_{c}", f"blank_line_ratio_{c}"]] = df_plag_test[c].apply(stylometric_analysis).apply(pd.Series)

In [14]:
for df in [df_plag_train, df_plag_val, df_plag_test]:
    for feat in ["comment_density", "avg_line_length", "line_length_variance", "blank_line_ratio"]:
        df[f"{feat}_delta"] = abs(df[f"{feat}_code1"] - df[f"{feat}_code2"])

In [15]:
df_plag_train.head()

,code1,code2,label,approx_tokens_code1,approx_tokens_code2,comment_density_code1,avg_line_length_code1,line_length_variance_code1,blank_line_ratio_code1,comment_density_code2,avg_line_length_code2,line_length_variance_code2,blank_line_ratio_code2,comment_density_delta,avg_line_length_delta,line_length_variance_delta,blank_line_ratio_delta
0,private void respawn(XmppAgent agent) {\r\...,"\tpublic FTPClient sample1a(String server, int...",0,140,26,0.0,44.933333,646.284444,0.042553,0.0,42.666667,1461.888889,0.000000,0.0,2.266667,815.604444,0.042553
1,public synchronized OutputStream getOutput...,"\tpublic static void copyFile3(File srcFile, F...",0,57,40,0.0,46.538462,1357.017751,0.000000,0.0,28.363636,516.231405,0.083333,0.0,18.174825,840.786347,0.083333
2,private String urlConnectionTranslate(Stri...,private String fetch(URL url) {\r\n ...,1,64,49,0.0,50.111111,1181.209877,0.000000,0.0,34.200000,512.293333,0.000000,0.0,15.911111,668.916543,0.000000
3,public static CodeBlock parse(BufferedRead...,public static CLocation convertSecondarySt...,1,264,43,0.0,46.974684,507.189232,0.000000,0.0,53.700000,3305.010000,0.000000,0.0,6.725316,2797.820768,0.000000
4,public static String MD5(String val) throw...,public static String encryptPassword(Strin...,1,20,76,0.0,52.000000,592.400000,0.000000,0.0,36.461538,407.633136,0.000000,0.0,15.538462,184.766864,0.000000


In [16]:
df_plag_train.describe().T

,count,mean,std,min,25%,50%,75%,max
label,24000.0,0.500000,5.000104e-01,0.000000,0.000000,0.500000,1.000000,1.000000e+00
approx_tokens_code1,24000.0,111.000542,1.893199e+02,4.000000,37.000000,64.000000,116.000000,4.285000e+03
approx_tokens_code2,24000.0,72.529958,1.176000e+02,7.000000,27.000000,42.000000,78.000000,3.494000e+03
comment_density_code1,24000.0,0.002363,2.422436e-02,0.000000,0.000000,0.000000,0.000000,7.647059e-01
avg_line_length_code1,24000.0,46.737325,1.691564e+02,14.306122,35.238095,41.384615,48.643433,1.093033e+04
line_length_variance_code1,24000.0,61664.666554,3.295064e+06,51.654321,411.794184,626.665864,965.871581,2.369385e+08
blank_line_ratio_code1,24000.0,0.009920,4.428539e-02,0.000000,0.000000,0.000000,0.000000,7.323944e-01
comment_density_code2,24000.0,0.006362,3.821806e-02,0.000000,0.000000,0.000000,0.000000,7.647059e-01
avg_line_length_code2,24000.0,38.943733,1.463895e+02,13.444444,31.250000,34.777778,40.333333,1.087867e+04
line_length_variance_code2,24000.0,43405.714051,3.003394e+06,45.264463,476.061224,633.551020,834.023669,2.346295e+08


In [17]:
stylometric_feats = df_ai_train["code"].apply(stylometric_analysis).apply(pd.Series)
df_ai_train = pd.concat([df_ai_train, stylometric_feats], axis=1)

In [18]:
stylometric_feats = df_ai_val["code"].apply(stylometric_analysis).apply(pd.Series)
df_ai_val = pd.concat([df_ai_val, stylometric_feats], axis=1)

In [19]:
stylometric_feats = df_ai_test["code"].apply(stylometric_analysis).apply(pd.Series)
df_ai_test = pd.concat([df_ai_test, stylometric_feats], axis=1)

In [20]:
df_ai_train.head()

,code,label,approx_tokens,comment_density,avg_line_length,line_length_variance,blank_line_ratio
0,private Set<Integer> perNodeRelease(final C th...,0,101,0.0,61.882353,1268.339100,0.055556
1,@Override\r\n public AuthenticationStatus f...,0,23,0.0,34.777778,702.395062,0.100000
2,public void callWorkListenerWithError(WorkCont...,1,28,0.0,49.500000,2072.250000,0.000000
3,public void setSubscription(Subscription s) {\...,1,14,0.0,33.250000,500.187500,0.000000
4,public boolean getDialogContentInset(int theme...,1,20,0.0,44.666667,1461.222222,0.000000


In [21]:
df_ai_train.describe().T

,count,mean,std,min,25%,50%,75%,max
label,24000.0,0.500000,0.500010,0.000000,0.000000,0.500000,1.000000,1.000000
approx_tokens,24000.0,46.552917,56.475617,6.000000,21.000000,33.000000,55.000000,3284.000000
comment_density,24000.0,0.032119,0.088502,0.000000,0.000000,0.000000,0.000000,0.789474
avg_line_length,24000.0,39.038119,11.381493,12.666667,31.222222,37.400000,45.000000,143.000000
line_length_variance,24000.0,788.535933,603.862741,0.000000,431.397135,641.660872,966.191645,21787.555556
blank_line_ratio,24000.0,0.044522,0.079357,0.000000,0.000000,0.000000,0.074074,0.849057


In [22]:
print(df_ai_train.groupby("label")["line_length_variance"].mean().round(2))

label
0    811.52
1    765.55
Name: line_length_variance, dtype: float64


In [23]:
print(df_ai_train.groupby("label")["comment_density"].mean().round(2))

label
0    0.00
1    0.06
Name: comment_density, dtype: float64


## 4. Semantic analysis

### 4.1 Set up parser

In [24]:
class ASTNodeType(Enum):
    PROGRAM = auto()
    CLASS = auto()
    METHOD = auto()
    IF = auto()
    FOR = auto()
    WHILE = auto()
    SWITCH = auto()
    BLOCK = auto()
    LITERAL = auto()
    ID = auto()
    OTHER = auto()

class ASTNode:
    def __init__(self, type: ASTNodeType, value: str = None):
        self.type     = type
        self.value    = value
        self.children = []

    def add_child(self, node):
        self.children.append(node)
        return node

    def __repr__(self):
        return f"Node({self.type.name}, {self.value!r}, children={len(self.children)})"

In [25]:
class Parser:
    def __init__(self, tokens: list):
        self.tokens = [t for t in tokens if t.type != TokenType.COMMENT]
        self.pos = 0

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None

    def consume(self):
        token = self.tokens[self.pos]
        self.pos += 1
        return token

    def match(self, ttype, value=None):
        token = self.peek()
        if token is None:
            return False
        if token.type != ttype:
            return False
        if value is not None and token.value != value:
            return False
        return True

    def parse(self):
        root = ASTNode(ASTNodeType.PROGRAM)
        while self.peek() is not None:
            node = self.parse_statement()
            if node:
                root.add_child(node)
        return root

    def parse_statement(self):
        token = self.peek()
        if token is None:
            return None

        if token.type == TokenType.KEYWORD and token.value == "class":
            return self.parse_class()

        # void/static/public ID (
        if token.type == TokenType.KEYWORD and token.value in {"void", "static", "public", "private", "protected"}:
            return self.parse_possible_method()

        if token.type == TokenType.KEYWORD and token.value == "if":
            return self.parse_control_flow(ASTNodeType.IF)
        if token.type == TokenType.KEYWORD and token.value == "for":
            return self.parse_control_flow(ASTNodeType.FOR)
        if token.type == TokenType.KEYWORD and token.value == "while":
            return self.parse_control_flow(ASTNodeType.WHILE)
        if token.type == TokenType.KEYWORD and token.value == "switch":
            return self.parse_control_flow(ASTNodeType.SWITCH)

        if self.match(TokenType.OPEN_BRACE):
            return self.parse_block()

        if token.type == TokenType.STRING:
            self.consume()
            return ASTNode(ASTNodeType.LITERAL, token.value)
        if token.type == TokenType.NUMBER:
            self.consume()
            return ASTNode(ASTNodeType.LITERAL, token.value)

        if token.type == TokenType.ID:
            self.consume()
            return ASTNode(ASTNodeType.ID, token.value)

        self.consume()
        return ASTNode(ASTNodeType.OTHER, token.value)

    def parse_class(self):
        self.consume()
        node = ASTNode(ASTNodeType.CLASS)
        if self.match(TokenType.ID):
            node.value = self.consume().value
        if self.match(TokenType.OPEN_BRACE):
            node.add_child(self.parse_block())
        return node

    def parse_possible_method(self):
        while self.peek() and self.peek().type == TokenType.KEYWORD:
            self.consume()
        # Confirm grammar rule is followed
        if (self.match(TokenType.ID)):
            name = self.consume().value
            if self.match(TokenType.OPEN_PAREN):
                node = ASTNode(ASTNodeType.METHOD, name)
                depth = 0
                while self.peek():
                    t = self.consume()
                    if t.type == TokenType.OPEN_PAREN:
                        depth += 1
                    elif t.type == TokenType.CLOSE_PAREN:
                        depth -= 1
                        if depth <= 0:
                            break
                if self.match(TokenType.OPEN_BRACE):
                    node.add_child(self.parse_block())
                return node
        return ASTNode(ASTNodeType.OTHER)

    def parse_control_flow(self, node_type):
        self.consume()
        node = ASTNode(node_type)
        # get conditional
        if self.match(TokenType.OPEN_PAREN):
            depth = 0
            while self.peek():
                t = self.consume()
                if t.type == TokenType.OPEN_PAREN:
                    depth += 1
                elif t.type == TokenType.CLOSE_PAREN:
                    depth -= 1
                    if depth <= 0:
                        break
        if self.match(TokenType.OPEN_BRACE):
            node.add_child(self.parse_block())
        return node

    def parse_block(self):
        self.consume()
        node = ASTNode(ASTNodeType.BLOCK)
        while self.peek() and not self.match(TokenType.CLOSE_BRACE):
            child = self.parse_statement()
            if child:
                node.add_child(child)
        if self.match(TokenType.CLOSE_BRACE):
            self.consume()
        return node

In [26]:
def print_ast(node, indent=0):
    print("  " * indent + repr(node))
    for child in node.children:
        print_ast(child, indent + 1)

In [31]:
def ast_analysis(code: str) -> dict:
    if not isinstance(code, str) or len(code.strip()) == 0:
        return {
            "num_classes": 0,
            "num_methods": 0,
            "num_if": 0,
            "num_for": 0,
            "num_while": 0,
            "num_switch": 0,
            "o_complexity": 1,
            "max_depth": 0,
            "total_nodes": 0,
            "num_literals": 0,
            "num_ids": 0,
            "unique_ids": 0,
            "id_diversity": 0,
        }

    tokens = lex(code)
    parser = Parser(tokens)
    ast = parser.parse()

    num_classes = 0
    num_methods = 0
    num_if = 0
    num_for = 0
    num_while = 0
    num_switch = 0
    num_literals = 0
    ids = []
    total_nodes = 0

    def walk(node, depth):
        num_classes, num_methods, num_if, num_for, num_while, num_switch, num_literals, total_nodes = 0, 0, 0, 0, 0, 0, 0, 0

        total_nodes += 1

        if node.type == ASTNodeType.CLASS:
            num_classes += 1
        elif node.type == ASTNodeType.METHOD:
            num_methods += 1
        elif node.type == ASTNodeType.IF:
            num_if += 1
        elif node.type == ASTNodeType.FOR:
            num_for += 1
        elif node.type == ASTNodeType.WHILE:
            num_while += 1
        elif node.type == ASTNodeType.SWITCH:
            num_switch += 1
        elif node.type == ASTNodeType.LITERAL:
            num_literals += 1
        elif node.type == ASTNodeType.ID and node.value:
            ids.append(node.value)

        if not node.children:
            return depth

        return max(walk(child, depth + 1) for child in node.children)

    max_depth = walk(ast, 0)

    # In case parser misses, check lexer tokens
    num_classes = num_classes or sum(
        1 for t in tokens
        if t.type == TokenType.KEYWORD and t.value == "class"
    )

    num_ids = len(ids)
    unique_ids = len(set(ids))
    id_diversity = (
        unique_ids / num_ids if num_ids > 0 else 0
    )

    o_complexity = num_if + num_for + num_while + num_switch + 1

    return {
        "num_classes": num_classes,
        "num_methods": num_methods,
        "num_if": num_if,
        "num_for": num_for,
        "num_while": num_while,
        "num_switch": num_switch,
        "o_complexity": o_complexity,
        "max_depth": max_depth,
        "total_nodes": total_nodes,
        "num_literals": num_literals,
        "num_ids": num_ids,
        "unique_ids": unique_ids,
        "id_diversity": id_diversity,
    }

### 4.2 Store features in dataframes

In [32]:
for c in ["code1", "code2"]:
    df_plag_train[[f"num_classes_{c}", f"num_methods_{c}", f"num_if_{c}", f"num_for_{c}", f"num_while_{c}", f"num_switch_{c}", f"o_complexity_{c}", f"max_depth_{c}", f"total_nodes_{c}", f"num_literals_{c}", f"num_ids_{c}", f"unique_ids_{c}", f"id_diversity_{c}"]] = df_plag_train[c].apply(ast_analysis).apply(pd.Series)
    df_plag_val[[f"num_classes_{c}", f"num_methods_{c}", f"num_if_{c}", f"num_for_{c}", f"num_while_{c}", f"num_switch_{c}", f"o_complexity_{c}", f"max_depth_{c}", f"total_nodes_{c}", f"num_literals_{c}", f"num_ids_{c}", f"unique_ids_{c}", f"id_diversity_{c}"]] = df_plag_val[c].apply(ast_analysis).apply(pd.Series)
    df_plag_test[[f"num_classes_{c}", f"num_methods_{c}", f"num_if_{c}", f"num_for_{c}", f"num_while_{c}", f"num_switch_{c}", f"o_complexity_{c}", f"max_depth_{c}", f"total_nodes_{c}", f"num_literals_{c}", f"num_ids_{c}", f"unique_ids_{c}", f"id_diversity_{c}"]] = df_plag_test[c].apply(ast_analysis).apply(pd.Series)

In [33]:
for df in [df_plag_train, df_plag_val, df_plag_test]:
    for feat in ["num_classes", "num_methods", "num_if", "num_for", "num_while", "num_switch", "o_complexity", "max_depth", "total_nodes", "num_literals", "num_ids", "unique_ids", "id_diversity"]:
        df[f"{feat}_delta"] = abs(df[f"{feat}_code1"] - df[f"{feat}_code2"])

In [34]:
df_plag_train.head()

,code1,code2,label,approx_tokens_code1,approx_tokens_code2,comment_density_code1,avg_line_length_code1,line_length_variance_code1,blank_line_ratio_code1,comment_density_code2,...,num_for_delta,num_while_delta,num_switch_delta,o_complexity_delta,max_depth_delta,total_nodes_delta,num_literals_delta,num_ids_delta,unique_ids_delta,id_diversity_delta
0,private void respawn(XmppAgent agent) {\r\...,"\tpublic FTPClient sample1a(String server, int...",0,140,26,0.0,44.933333,646.284444,0.042553,0.0,...,0.0,0.0,0.0,0.0,11.0,0.0,0.0,90.0,49.0,0.030702
1,public synchronized OutputStream getOutput...,"\tpublic static void copyFile3(File srcFile, F...",0,57,40,0.0,46.538462,1357.017751,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,9.0,0.085598
2,private String urlConnectionTranslate(Stri...,private String fetch(URL url) {\r\n ...,1,64,49,0.0,50.111111,1181.209877,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.0,10.0,0.062112
3,public static CodeBlock parse(BufferedRead...,public static CLocation convertSecondarySt...,1,264,43,0.0,46.974684,507.189232,0.000000,0.0,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,138.0,43.0,0.114335
4,public static String MD5(String val) throw...,public static String encryptPassword(Strin...,1,20,76,0.0,52.000000,592.400000,0.000000,0.0,...,0.0,0.0,0.0,0.0,5.0,0.0,0.0,24.0,14.0,0.085679


In [35]:
df_plag_train.describe().T

,count,mean,std,min,25%,50%,75%,max
label,24000.0,0.500000,5.000104e-01,0.000000,0.000000,0.500000,1.000000,1.000000e+00
approx_tokens_code1,24000.0,111.000542,1.893199e+02,4.000000,37.000000,64.000000,116.000000,4.285000e+03
approx_tokens_code2,24000.0,72.529958,1.176000e+02,7.000000,27.000000,42.000000,78.000000,3.494000e+03
comment_density_code1,24000.0,0.002363,2.422436e-02,0.000000,0.000000,0.000000,0.000000,7.647059e-01
avg_line_length_code1,24000.0,46.737325,1.691564e+02,14.306122,35.238095,41.384615,48.643433,1.093033e+04
line_length_variance_code1,24000.0,61664.666554,3.295064e+06,51.654321,411.794184,626.665864,965.871581,2.369385e+08
blank_line_ratio_code1,24000.0,0.009920,4.428539e-02,0.000000,0.000000,0.000000,0.000000,7.323944e-01
comment_density_code2,24000.0,0.006362,3.821806e-02,0.000000,0.000000,0.000000,0.000000,7.647059e-01
avg_line_length_code2,24000.0,38.943733,1.463895e+02,13.444444,31.250000,34.777778,40.333333,1.087867e+04
line_length_variance_code2,24000.0,43405.714051,3.003394e+06,45.264463,476.061224,633.551020,834.023669,2.346295e+08


In [36]:
ast_feats = df_ai_train["code"].apply(ast_analysis).apply(pd.Series)
df_ai_train = pd.concat([df_ai_train, ast_feats], axis=1)

In [37]:
ast_feats = df_ai_val["code"].apply(ast_analysis).apply(pd.Series)
df_ai_val = pd.concat([df_ai_val, ast_feats], axis=1)

In [38]:
ast_feats = df_ai_test["code"].apply(ast_analysis).apply(pd.Series)
df_ai_test = pd.concat([df_ai_test, ast_feats], axis=1)

In [39]:
df_ai_train.head()

,code,label,approx_tokens,comment_density,avg_line_length,line_length_variance,blank_line_ratio,num_classes,num_methods,num_if,num_for,num_while,num_switch,o_complexity,max_depth,total_nodes,num_literals,num_ids,unique_ids,id_diversity
0,private Set<Integer> perNodeRelease(final C th...,0,101,0.0,61.882353,1268.339100,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,0.0,63.0,30.0,0.476190
1,@Override\r\n public AuthenticationStatus f...,0,23,0.0,34.777778,702.395062,0.100000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,19.0,15.0,0.789474
2,public void callWorkListenerWithError(WorkCont...,1,28,0.0,49.500000,2072.250000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,7.0,7.0,1.000000
3,public void setSubscription(Subscription s) {\...,1,14,0.0,33.250000,500.187500,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,6.0,5.0,0.833333
4,public boolean getDialogContentInset(int theme...,1,20,0.0,44.666667,1461.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,21.0,16.0,0.761905


In [40]:
print(df_ai_train.groupby("label")["max_depth"].mean().round(2))

label
0    4.32
1    4.09
Name: max_depth, dtype: float64


In [41]:
print(df_ai_train.groupby("label")["o_complexity"].mean().round(2))

label
0    1.0
1    1.0
Name: o_complexity, dtype: float64


### 4.3 Update CSVs with new columns

In [42]:
df_plag_train.to_csv("df_plagiarism_train.csv", index=False)
df_plag_val.to_csv("df_plagiarism_val.csv", index=False)
df_plag_test.to_csv("df_plagiarism_test.csv", index=False)

In [43]:
df_ai_train.to_csv("df_ai_train.csv", index=False)
df_ai_val.to_csv("df_ai_val.csv", index=False)
df_ai_test.to_csv("df_ai_test.csv", index=False)